# Train AC-MPC and AC-MLP for gate racing on Colab

Trains both policies of the ACMPC paper (Romero et al., T-RO 2025) on the drone-racing task in `acmpc/race_env.py`, with the paper's PPO hyperparameters (Table II). The environment is plain numpy, so nothing needs compiling. The differentiable MPC runs on the GPU.

**VS Code:** open this notebook, **Select Kernel → Colab → New Colab Server → GPU (T4 or better)**, then **Run All**.

Afterwards, download `race_runs.zip`, unzip it into `acmpc/saved/race/` on the Mac, and evaluate or fly the policies there (see the last cell).

In [ ]:
!nvidia-smi -L || echo "No GPU: switch the runtime to a GPU, AC-MPC is very slow on CPU"

## Setup

In [ ]:
%cd /content
!test -d agile-flight-macos || git clone -q https://github.com/krmanik/agile-flight-macos.git
!cd agile-flight-macos && git pull -q
!test -d acmpc_public || git clone -q https://github.com/uzh-rpg/acmpc_public.git
!cd acmpc_public && git -c url.https://github.com/.insteadOf=git@github.com: submodule update -q --init mpc.pytorch
!cd acmpc_public && git checkout -q -- training_modules && git apply ../agile-flight-macos/patches/acmpc_public.patch
!cd acmpc_public/mpc.pytorch && git checkout -q -- mpc && git apply ../../agile-flight-macos/patches/mpc_pytorch.patch
!pip -q install "stable-baselines3>=2.4" gymnasium setproctitle
import os
os.environ["ACMPC_PATH"] = "/content/acmpc_public"
%cd /content/agile-flight-macos

## Settings

Runs are saved to Google Drive when it can be mounted (survives a disconnect), otherwise to `/content/race_runs`. Re-running the training cell resumes existing runs from their last checkpoint.

On a T4, AC-MLP takes ~2 s per iteration (~1 h for 2000) and AC-MPC ~30 s (~17 h), longer than a Colab session. AC-MPC therefore needs several sessions, and its runs must survive a disconnect. If the Settings cell says "Google Drive not mounted" (from the VS Code Colab extension it failed with "User cancelled dfs_ephemeral authorization"), run AC-MPC in the browser instead: colab.research.google.com → File → Open notebook → GitHub → `krmanik/agile-flight-macos`, this notebook, GPU runtime, and allow the Drive access.

In [ ]:
import glob

TRACK = "splits"        # acmpc/tracks/: splits (paper), circle
ITERATIONS = 2000       # paper: ~2000 PPO iterations of 25k samples (50M steps)
POLICIES = ["mpc", "mlp"]

LOG_DIR = "/content/race_runs"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/acmpc_race_runs"
except Exception as e:
    print("Google Drive not mounted, saving to", LOG_DIR, "-", e)
os.makedirs(LOG_DIR, exist_ok=True)
print("Runs in", LOG_DIR)

def latest_run(pol):
    runs = glob.glob(f"{LOG_DIR}/{pol.upper()}_{TRACK}_*")
    return max(runs, key=lambda r: int(r.rsplit("_", 1)[1]), default=None)

def latest_iter(run):
    return max((int(os.path.basename(p)[5:10]) for p in glob.glob(run + "/Policy/*.pth")), default=0)

## Train both policies in parallel

In [ ]:
import signal, subprocess, time

# One training process per policy: two AC-MPC runs (~7 GB each) do not fit on a
# T4 together. Stop the ones started by an earlier run of this cell first;
# SIGINT lets them save a checkpoint, which the restart resumes from.
old = subprocess.run(["pgrep", "-f", "acmpc/train_race.py"], capture_output=True, text=True).stdout.split()
for pid in old:
    os.kill(int(pid), signal.SIGINT)
for _ in range(120):
    if not subprocess.run(["pgrep", "-f", "acmpc/train_race.py"], capture_output=True).stdout:
        break
    time.sleep(1)
else:
    subprocess.run(["pkill", "-9", "-f", "acmpc/train_race.py"])
print("Stopped", len(old), "running training process(es)")

env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
procs = {}
for pol in POLICIES:
    run = latest_run(pol)
    args = ["python", "acmpc/train_race.py", "--iterations", str(ITERATIONS)]
    if run and latest_iter(run) >= ITERATIONS:
        print(pol, "finished:", run)
        continue
    if run and latest_iter(run):
        args += ["--resume", run]
    else:
        args += ["--policy", pol, "--track", TRACK, "--log_dir", LOG_DIR]
    log = open(f"/content/train_{pol}.log", "a")
    # Own session: interrupting a notebook cell must not Ctrl-C the training
    procs[pol] = subprocess.Popen(args, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True)
    print(pol, " ".join(args))

Progress (re-run any time; stopping this cell does not stop the training). Re-running the cell above restarts the training from the latest checkpoints:

In [ ]:
import re, time
from IPython.display import clear_output

def last(log, key):
    # Only the newest process: the log is appended to on every restart
    text = open(log).read().rsplit("Run directory:", 1)[-1]
    m = re.findall(rf"\|\s+{key}\s+\|\s+([-\d.e+]+)", text)
    return m[-1] if m else "-"

while any(p.poll() is None for p in procs.values()):
    clear_output(wait=True)
    for pol, p in procs.items():
        log = f"/content/train_{pol}.log"
        print(f"{pol.upper()}: iteration {last(log, 'iteration')}/{ITERATIONS}  "
              f"success {last(log, 'success_rate')}  gates {last(log, 'gates_passed')}  "
              f"ep_rew {last(log, 'ep_rew_mean')}  {last(log, 'iteration_s')} s/it  "
              f"{'running' if p.poll() is None else 'exit ' + str(p.returncode)}")
    time.sleep(60)
for pol in procs:
    print(pol, "done:"); print(open(f"/content/train_{pol}.log").read()[-1500:])

Optional: TensorBoard (`race/success_rate`, `race/gates_passed`, `rollout/ep_rew_mean`).

In [ ]:
%load_ext tensorboard
%tensorboard --logdir $LOG_DIR

## Compare AC-MPC vs AC-MLP

In [ ]:
runs = [latest_run(p) for p in POLICIES]
RUNS = " ".join(runs)
!python acmpc/eval_race.py $RUNS --plot /content/race.png
!python acmpc/eval_race.py $RUNS --sweep mass_scale 0.8 1.0 1.2 1.4 --episodes 32
from IPython.display import Image
Image("/content/race.png")

## Package the runs for the Mac (latest checkpoint + TensorBoard logs)

In [ ]:
import shutil, zipfile
runs = [latest_run(p) for p in POLICIES]
zpath = "/content/race_runs.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for run in runs:
        name = os.path.basename(run)
        it = latest_iter(run)
        files = [run + "/config.json", f"{run}/Policy/iter_{it:05d}.pth", f"{run}/RMS/iter_{it:05d}.npz"]
        files += glob.glob(run + "/tb_*/*")
        for f in files:
            z.write(f, os.path.join(name, os.path.relpath(f, run)))
print(zpath, os.path.getsize(zpath) // 1024, "KB")
try:
    from google.colab import files
    files.download(zpath)
except Exception as e:
    print("Automatic download unavailable here:", e)
    print("Get it from Google Drive (copied next to the runs) or the Colab file browser.")
if LOG_DIR.startswith("/content/drive"):
    shutil.copy(zpath, LOG_DIR)

## On the Mac

```bash
unzip race_runs.zip -d acmpc/saved/race/
pixi run eval-race acmpc/saved/race/MPC_splits_1 acmpc/saved/race/MLP_splits_1 --plot race.png
pixi run race-sim --run acmpc/saved/race/MPC_splits_1 --render     # ROS sim + Unity, gates in rviz
```